# PV & Wind Power Curve Modeling

Erick Chauke

Physics-informed and data-driven power curve models for two PV plants and two wind plants, fit
from source power-curve grids (input files, gitignored). Six standalone models: PV1, PV2,
PV-combined, Wind1, Wind2, Wind-combined.

## Setup

Imports and one config cell for the source path and adjustable parameters.

### Imports

Numeric, tabular, plotting, and Excel-reading libraries used throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl

### Configuration

File path and adjustable parameters used by every later section.

In [ ]:
# Config: file path and adjustable params. Capacity values sourced from each sheet's
# title cell (row 0, col 0), e.g. "Installed Capacity: 75 MW".
DATA_PATH = "data/FPCs.xlsm"

PV_IRRADIANCE_STEP = 50    # W/m^2
PV_TEMP_STEP = 5           # deg C
WIND_VELOCITY_STEP = 0.5   # m/s
WIND_DIRECTION_STEP = 15   # deg

CV_FOLDS = 5

CAPACITY_MW = {
    "PV1": 75.0,
    "PV2": 75.0,
    "Wind1": 102.0,
    "Wind2": 86.6,
}

## Data Loading & Grid Parsing

Each sheet is a 2D power-curve grid, parsed by position (not pandas' auto-detected headers, which the merged/label cells break).

### Grid parsing helper

Positionally slices a sheet into capacity label, x-axis, y-axis, and power grid.

In [ ]:
def parse_grid_sheet(path, sheet_name):
    """Positional parse of one FPCs.xlsm-style grid sheet into title, x-axis, y-axis, power grid."""
    df = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")
    capacity_label = df.iloc[0, 0]
    x_axis = df.iloc[1, 2:].astype(float).to_numpy()
    y_axis = df.iloc[2:, 1].astype(float).to_numpy()
    power_grid = df.iloc[2:, 2:].astype(float).to_numpy()
    return capacity_label, x_axis, y_axis, power_grid

### Load all four sheets

Applies the helper to PV1, PV2, Wind1, and Wind2 into one dict keyed by sheet name.

In [ ]:
grids = {}
for sheet_name in ["PV1", "PV2", "Wind1", "Wind2"]:
    capacity_label, x_axis, y_axis, power_grid = parse_grid_sheet(DATA_PATH, sheet_name)
    grids[sheet_name] = {
        "capacity_label": capacity_label,
        "x_axis": x_axis,
        "y_axis": y_axis,
        "power_grid": power_grid,
    }

### Sanity check

Confirms shapes and axis ranges for all four parsed grids.

In [ ]:
for sheet_name, g in grids.items():
    print(
        sheet_name, "|", g["capacity_label"],
        "| grid shape:", g["power_grid"].shape,
        "| x range:", g["x_axis"].min(), "-", g["x_axis"].max(),
        "| y range:", g["y_axis"].min(), "-", g["y_axis"].max(),
    )